In [ ]:
import pandas as pd
import os

rmm_df = pd.read_csv("/orcd/data/satra/001/users/brukew/sailsprep/subset_data/RMM.csv")

## Render RMM videos from given COI target directories

In [ ]:
def get_file_name(video_info: dict) -> str:
    id = video_info['ID']
    filename = video_info['FileName']
    coder = video_info['Original_Coder']
    
    return f"{id}_{coder}_{filename}"

def get_target(target_path: str):
    """
    Get target track ID from result.json in target_path.
    """
    import json
    result = target_path + "/result.json"
    if not os.path.exists(result):
        raise FileNotFoundError(f"Result file not found at {result}")
    json_data = json.load(open(result))
    if 'track_id' in json_data:
        return json_data['track_id']
    else:
        if 'reason' in json_data:
            print(f"Target data not found for {target_path}: {json_data['reason']}")
        return None
    
def load_track_from_h5(h5_path: Path, video_path: Optional[Path] = None) -> LoadedTrack:
    """Load a single tracking HDF5 file into a `Track` dataclass."""
    with h5py.File(str(h5_path), "r") as f:
        metadata = f["metadata"]
        start_frame = int(metadata.attrs.get("start_frame", 0))
        end_frame = int(metadata.attrs.get("end_frame", start_frame))
        fps = float(metadata.attrs.get("video_fps", 0.0) or 0.0)
        num_frames_attr = int(metadata.attrs.get("num_frames", 0))
        track_id_attr = metadata.attrs.get("track_id")

        if track_id_attr is None:
            # Fallback to file name (e.g., track_0007.h5)
            stem = h5_path.stem
            try:
                track_id_attr = int(stem.split("_")[-1])
            except Exception:
                track_id_attr = -1

        frames = f["frames"]
        frame_entries = sorted(frames.keys())

        frame_numbers: List[int] = []
        bboxes: List[Optional[tuple]] = []
        keypoints: List[Optional[list]] = []

        for frame_key in frame_entries:
            try:
                frame_number = int(frame_key.split("_")[-1])
            except Exception:
                frame_number = len(frame_numbers)

            frame_numbers.append(frame_number)

            frame = frames[frame_key]
            frame_data = _load_frame_group(frame)

            bboxes.append(frame_data.get("bbox"))
            keypoints.append(frame_data.get("keypoints"))

        track = Track(
            id=int(track_id_attr),
            start_frame=start_frame,
            end_frame=end_frame,
            fps=fps,
            keypoints=keypoints,
            bboxes=bboxes,
            face_crops=None,
            video_path=str(video_path) if video_path else None,
            frame_numbers=frame_numbers,
            meta={
                "num_frames": num_frames_attr or len(frame_numbers),
                "video_width": metadata.attrs.get("video_width"),
                "video_height": metadata.attrs.get("video_height"),
                "source_h5": str(h5_path),
            },
        )

    return LoadedTrack(track=track, h5_path=h5_path)

def render_vid(video_info, target_path, tracking_path=None):
    if not tracking_path:
        raise ValueError("Tracking path is required for rendering video.")
    target_id = get_target(target_path)
    if not target_id:
        print(f"No target ID found for {target_path}. Skipping rendering.")
        return

    
    
    ## get result.json from target

    ## get track data from tracking

    ## render video with track data overlayed on video from video_info

### Setup 

In [ ]:
DIRS = []
TRACKING_DIRS = [dir + "/tracking" for dir in DIRS]
TARGET_DIRS = [dir + "/targets" for dir in DIRS]

targets = {os.path.basename(file_path): dir for dir in TARGET_DIRS for file_path in os.listdir(dir)}
tracking = {os.path.basename(file_path): dir for dir in TRACKING_DIRS for file_path in os.listdir(dir)}


### Collect Videos

In [ ]:
for row, video_info in rmm_df.iterrows():
    file_name = get_file_name(video_info)
    target_name = file_name + "_target"
    tracking_name = file_name + "_tracking"
    if target_name in targets:
        target_path = targets[target_name] + "/" + target_name
        tracking_dir = tracking.get(tracking_name, None)
        if tracking_dir:
            tracking_path = tracking_dir + "/" + tracking_name
        else:
            tracking_path = None
        render_vid(video_info, target_path, tracking_path)
    